In [4]:
!nvidia-smi
!python --version

Sat Sep 26 23:32:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from kaggle_secrets import UserSecretsClient
import os
import subprocess
from pathlib import Path

repo = "/kaggle/working/do-an-ptit-d22"

user_secrets = UserSecretsClient()
token = user_secrets.get_secret("GH_PAT")

askpass = Path("/kaggle/working/git-askpass.sh")
askpass.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
"""
)
askpass.chmod(0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"

subprocess.run(
    [
        "git",
        "clone",
        "--branch",
        "feat/m1-02-concurrency-sweep-baseline",
        "--single-branch",
        "https://github.com/theflyingkhui04/do-an-ptit-d22.git",
        repo,
    ],
    env=env,
    check=True,
)

askpass.unlink(missing_ok=True)

print("Clone OK")

Cloning into '/kaggle/working/do-an-ptit-d22'...


Clone OK


In [6]:
!git -C /kaggle/working/do-an-ptit-d22 rev-parse HEAD
!git -C /kaggle/working/do-an-ptit-d22 status --short
!git -C /kaggle/working/do-an-ptit-d22 log -1 --oneline

11064f15cb8d52e6d58d3612c6a535a8e4428adf
11064f1 (HEAD -> feat/m1-02-concurrency-sweep-baseline, origin/feat/m1-02-concurrency-sweep-baseline) feat(m1-02): add concurrency-matched warmup


In [7]:
!grep -n \
  -e "effective_warmup_count" \
  -e "warmup_concurrency" \
  -e "option-b-concurrency-matched" \
  /kaggle/working/do-an-ptit-d22/experiments/benchmark_harness.py

403:    effective_warmup_count = max(
408:    warmup_concurrency = args.concurrency
411:    print("policy               : option-b-concurrency-matched")
414:    print(f"warmup_requests      : {effective_warmup_count}")
415:    print(f"warmup_concurrency   : {warmup_concurrency}")
425:                count=effective_warmup_count,
426:                concurrency=warmup_concurrency,
458:            print(f"concurrency_limit    : {warmup_concurrency}")


In [8]:
import torch
import importlib.metadata as m

print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("compute capability:", torch.cuda.get_device_capability(0))

for package in ["vllm", "aiohttp", "nvidia-ml-py"]:
    try:
        print(package, m.version(package))
    except m.PackageNotFoundError:
        print(package, "NOT INSTALLED")

torch: 2.13.0+cu130
torch CUDA: 13.0
CUDA available: True
GPU: Tesla T4
compute capability: (7, 5)
vllm 0.29.0
aiohttp 3.13.5
nvidia-ml-py 13.595.45


In [9]:
!python -m pip check

bigframes 2.39.0 requires google-cloud-bigquery-storage, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage, which is not installed.
grpcio-tools 1.71.2 has requirement protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6.
ydata-profiling 4.18.4 has requirement numba<0.63,>=0.60, but you have numba 0.65.0.
google-colab 1.0.0 has requirement jupyter-server==2.14.0, but you have jupyter-server 2.12.5.
google-colab 1.0.0 has requirement pandas==2.2.2, but you have pandas 2.3.3.
dask-cuda 26.2.0 has requirement cuda-core==0.3.*, but you have cuda-core 1.2.1.
dopamine-rl 4.1.2 has requirement gym<=0.25.2, but you have gym 0.26.2.
moviepy 1.0.3 has requirement decorator<5.0,>=4.0.2, but you have decorator 5.3.1.
pylibcudf-cu12 26.2.1 has requirement cuda-python<13.0,>=12.9.2, but you have cuda-python 13.4.1.
numba-cuda 0.22.2 has requirement cuda-core<1.0.0,>=0.3.2, but you have cuda-core 1.2.1.
libcuml-cu12 26.2.0 has requirement cuda-toolkit[cublas,cufft,cur

In [10]:
!cd /kaggle/working/do-an-ptit-d22 && \
python -m py_compile experiments/benchmark_harness.py

In [12]:
import os
import subprocess

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"

log_file = open("/kaggle/working/vllm_server.log", "w")

server_process = subprocess.Popen(
    [
        "vllm",
        "serve",
        "Qwen/Qwen2.5-7B-Instruct-AWQ",
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
    ],
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("PID:", server_process.pid)

PID: 236


In [14]:
!tail -n 80 /kaggle/working/vllm_server.log

(APIServer pid=229) INFO 09-26 23:33:06 [api_utils.py:347] 
(APIServer pid=229) INFO 09-26 23:33:06 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=229) INFO 09-26 23:33:06 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.29.0
(APIServer pid=229) INFO 09-26 23:33:06 [api_utils.py:347]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-7B-Instruct-AWQ
(APIServer pid=229) INFO 09-26 23:33:06 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=229) INFO 09-26 23:33:06 [api_utils.py:347] 
(APIServer pid=229) INFO 09-26 23:33:06 [api_utils.py:286] non-default args: {'model_tag': 'Qwen/Qwen2.5-7B-Instruct-AWQ', 'host': '127.0.0.1', 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}
(APIServer pid=229) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


In [23]:
!curl -sS -o /dev/null -w "%{http_code}\n" \
  http://127.0.0.1:8000/health

200


In [24]:
!curl -sS http://127.0.0.1:8000/v1/models | python -m json.tool

{
    "object": "list",
    "data": [
        {
            "id": "Qwen/Qwen2.5-7B-Instruct-AWQ",
            "object": "model",
            "created": 1790426366,
            "owned_by": "vllm",
            "root": "Qwen/Qwen2.5-7B-Instruct-AWQ",
            "parent": null,
            "max_model_len": 32768,
            "permission": [
                {
                    "id": "modelperm-864333dabde2f4b8",
                    "object": "model_permission",
                    "created": 1790426366,
                    "allow_create_engine": false,
                    "allow_sampling": true,
                    "allow_logprobs": true,
                    "allow_search_indices": false,
                    "allow_view": true,
                    "allow_fine_tuning": false,
                    "organization": "*",
                    "group": null,
                    "is_blocking": false
                }
            ]
        }
    ]
}


In [25]:
%%bash

curl -sS http://127.0.0.1:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen2.5-7B-Instruct-AWQ",
    "messages": [
      {
        "role": "user",
        "content": "Trả lời đúng một từ: thủ đô của Việt Nam là gì?"
      }
    ],
    "temperature": 0,
    "max_tokens": 16
  }' | python -m json.tool

{
    "id": "chatcmpl-b34e80430576e47b",
    "object": "chat.completion",
    "created": 1790426369,
    "model": "Qwen/Qwen2.5-7B-Instruct-AWQ",
    "choices": [
        {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": "H\u00e0 N\u1ed9i",
                "refusal": null,
                "annotations": null,
                "audio": null,
                "function_call": null,
                "reasoning": null
            },
            "logprobs": null,
            "finish_reason": "stop",
            "stop_reason": null,
            "token_ids": null,
            "routed_experts": null
        }
    ],
    "service_tier": null,
    "system_fingerprint": "vllm-0.29.0-3e2a96bd",
    "usage": {
        "prompt_tokens": 44,
        "total_tokens": 48,
        "completion_tokens": 4,
        "prompt_tokens_details": null,
        "completion_tokens_details": null
    },
    "prompt_logprobs": null,
    "prompt_token_ids": n

# **TEST 1 — Option B tại c4**

In [26]:
!rm -f \
  /kaggle/working/do-an-ptit-d22/experiments/runs/option_b_c4.jsonl

!cd /kaggle/working/do-an-ptit-d22 && \
python experiments/benchmark_harness.py \
  --model Qwen/Qwen2.5-7B-Instruct-AWQ \
  --num-requests 10 \
  --concurrency 4 \
  --warmup-requests 3 \
  --max-tokens 32 \
  --lane vanilla \
  --workload smoke \
  --experiment-id exp-m1-02-option-b-c4-001 \
  --repeat-index 1 \
  --raw-out experiments/runs/option_b_c4.jsonl \
  --overwrite


===== ENVIRONMENT CONTRACT PASS =====
profile              : m1-t4-cu130-v1
gpu                  : Tesla T4
vllm_version         : 0.29.0
torch_version        : 2.13.0+cu130
torch_cuda_version   : 13.0

===== M1-02 WARM-UP POLICY =====
policy               : option-b-concurrency-matched
cell_id              : vanilla__smoke__c4
requested_minimum    : 3
warmup_requests      : 4
warmup_concurrency   : 4
measured_concurrency : 4

===== WARM-UP PASS =====
requests             : 4
successes            : 4
concurrency_limit    : 4

===== M1-02 BENCHMARK RESULT =====
experiment_id       : exp-m1-02-option-b-c4-001
cell_id             : vanilla__smoke__c4
repeat_id           : r001
run_id              : run-6e3df993-a32d-451c-b894-6dfd0540bd33
requests            : 10
successes           : 10
metrics_complete    : 10
concurrency         : 4
duration_s          : 0.926
request_throughput  : 10.794 req/s
output_throughput   : 118.732 tok/s
run_vram_peak_mb    : 13539.19
raw_jsonl           : ex

In [27]:
import json

path = (
    "/kaggle/working/do-an-ptit-d22/"
    "experiments/runs/option_b_c4.jsonl"
)

with open(path, encoding="utf-8") as f:
    rows = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("rows             :", len(rows))
print("successes        :", sum(r["success"] for r in rows))
print("metrics_complete :", sum(r["metrics_complete"] for r in rows))

print("experiment_ids   :", {r["experiment_id"] for r in rows})
print("cell_ids         :", {r["cell_id"] for r in rows})
print("repeat_ids       :", {r["repeat_id"] for r in rows})
print("phases           :", {r["phase"] for r in rows})
print("request_indices  :", [r["request_index"] for r in rows])

assert len(rows) == 10

assert all(r["success"] for r in rows)
assert all(r["metrics_complete"] for r in rows)
assert all(r["http_status"] == 200 for r in rows)

assert {r["experiment_id"] for r in rows} == {
    "exp-m1-02-option-b-c4-001"
}

assert {r["cell_id"] for r in rows} == {
    "vanilla__smoke__c4"
}

assert {r["repeat_id"] for r in rows} == {
    "r001"
}

# Critical: warm-up MUST NOT leak into measured raw data.
assert {r["phase"] for r in rows} == {
    "measured"
}

assert [r["request_index"] for r in rows] == list(range(10))

assert len({r["request_id"] for r in rows}) == 10
assert len({r["run_id"] for r in rows}) == 1

print("OPTION B C4 SMOKE: PASS")

rows             : 10
successes        : 10
metrics_complete : 10
experiment_ids   : {'exp-m1-02-option-b-c4-001'}
cell_ids         : {'vanilla__smoke__c4'}
repeat_ids       : {'r001'}
phases           : {'measured'}
request_indices  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
OPTION B C4 SMOKE: PASS


# **TEST 2 — CLI minimum lớn hơn concurrency**

In [28]:
!cd /kaggle/working/do-an-ptit-d22 && \
python experiments/benchmark_harness.py \
  --model Qwen/Qwen2.5-7B-Instruct-AWQ \
  --num-requests 5 \
  --concurrency 2 \
  --warmup-requests 5 \
  --max-tokens 16 \
  --lane vanilla \
  --workload smoke \
  --experiment-id exp-m1-02-option-b-c2-min5 \
  --repeat-index 1 \
  --raw-out experiments/runs/option_b_c2_min5.jsonl \
  --overwrite


===== ENVIRONMENT CONTRACT PASS =====
profile              : m1-t4-cu130-v1
gpu                  : Tesla T4
vllm_version         : 0.29.0
torch_version        : 2.13.0+cu130
torch_cuda_version   : 13.0

===== M1-02 WARM-UP POLICY =====
policy               : option-b-concurrency-matched
cell_id              : vanilla__smoke__c2
requested_minimum    : 5
warmup_requests      : 5
warmup_concurrency   : 2
measured_concurrency : 2

===== WARM-UP PASS =====
requests             : 5
successes            : 5
concurrency_limit    : 2

===== M1-02 BENCHMARK RESULT =====
experiment_id       : exp-m1-02-option-b-c2-min5
cell_id             : vanilla__smoke__c2
repeat_id           : r001
run_id              : run-7c3d49e7-0515-4a04-8430-16725d149061
requests            : 5
successes           : 5
metrics_complete    : 5
concurrency         : 2
duration_s          : 0.888
request_throughput  : 5.632 req/s
output_throughput   : 61.954 tok/s
run_vram_peak_mb    : 13539.19
raw_jsonl           : experi

# **TEST 3 — Protocol floor không thể bị tắt bằng 0**

In [29]:
!cd /kaggle/working/do-an-ptit-d22 && \
python experiments/benchmark_harness.py \
  --model Qwen/Qwen2.5-7B-Instruct-AWQ \
  --num-requests 1 \
  --concurrency 1 \
  --warmup-requests 0 \
  --max-tokens 16 \
  --lane vanilla \
  --workload smoke \
  --experiment-id exp-m1-02-option-b-floor \
  --repeat-index 1 \
  --raw-out experiments/runs/option_b_floor.jsonl \
  --overwrite


===== ENVIRONMENT CONTRACT PASS =====
profile              : m1-t4-cu130-v1
gpu                  : Tesla T4
vllm_version         : 0.29.0
torch_version        : 2.13.0+cu130
torch_cuda_version   : 13.0

===== M1-02 WARM-UP POLICY =====
policy               : option-b-concurrency-matched
cell_id              : vanilla__smoke__c1
requested_minimum    : 0
warmup_requests      : 3
warmup_concurrency   : 1
measured_concurrency : 1

===== WARM-UP PASS =====
requests             : 3
successes            : 3
concurrency_limit    : 1

===== M1-02 BENCHMARK RESULT =====
experiment_id       : exp-m1-02-option-b-floor
cell_id             : vanilla__smoke__c1
repeat_id           : r001
run_id              : run-78331746-7e0d-45c6-8fbd-4165e9a722b0
requests            : 1
successes           : 1
metrics_complete    : 1
concurrency         : 1
duration_s          : 0.292
request_throughput  : 3.422 req/s
output_throughput   : 37.644 tok/s
run_vram_peak_mb    : 13539.19
raw_jsonl           : experime

# **TEST 4 — Warm-up failure phải chặn measurement**

In [30]:
%%bash

cd /kaggle/working/do-an-ptit-d22

rm -f experiments/runs/should_not_exist.jsonl

set +e

python experiments/benchmark_harness.py \
  --endpoint http://127.0.0.1:8999/v1/chat/completions \
  --model Qwen/Qwen2.5-7B-Instruct-AWQ \
  --num-requests 4 \
  --concurrency 4 \
  --warmup-requests 3 \
  --max-tokens 16 \
  --lane vanilla \
  --workload smoke \
  --experiment-id exp-m1-02-warmup-failure \
  --repeat-index 1 \
  --raw-out experiments/runs/should_not_exist.jsonl \
  --overwrite

CODE=$?

echo "EXIT_CODE=$CODE"

if [ "$CODE" -ne 4 ]; then
    echo "FAIL: expected exit code 4"
    exit 1
fi

if [ -f experiments/runs/should_not_exist.jsonl ]; then
    echo "FAIL: measured raw file should not exist"
    exit 1
fi

echo "WARM-UP FAILURE GATE: PASS"


===== ENVIRONMENT CONTRACT PASS =====
profile              : m1-t4-cu130-v1
gpu                  : Tesla T4
vllm_version         : 0.29.0
torch_version        : 2.13.0+cu130
torch_cuda_version   : 13.0

===== M1-02 WARM-UP POLICY =====
policy               : option-b-concurrency-matched
cell_id              : vanilla__smoke__c4
requested_minimum    : 3
warmup_requests      : 4
warmup_concurrency   : 4
measured_concurrency : 4

===== WARM-UP FAILED =====
attempted            : 4
failed               : 4
failure              : request_id=req-4987baca-121b-4540-a8c9-b185c1d4ac94 error_type=ClientConnectorError error_message=Cannot connect to host 127.0.0.1:8999 ssl:default [Connect call failed ('127.0.0.1', 8999)]
failure              : request_id=req-ba9669f1-da30-41d9-905b-02a2526e255e error_type=ClientConnectorError error_message=Cannot connect to host 127.0.0.1:8999 ssl:default [Connect call failed ('127.0.0.1', 8999)]
failure              : request_id=req-96d3a7d4-2b31-4c1b-b6f9-ce8

# **TEST 5 — Verify exact Git SHA được ghi vào raw**

In [31]:
import json
import subprocess

repo = "/kaggle/working/do-an-ptit-d22"

expected_sha = subprocess.check_output(
    ["git", "-C", repo, "rev-parse", "HEAD"],
    text=True,
).strip()

path = f"{repo}/experiments/runs/option_b_c4.jsonl"

with open(path, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

recorded_shas = {r["commit"] for r in rows}

print("expected SHA :", expected_sha)
print("recorded SHA :", recorded_shas)

assert recorded_shas == {expected_sha}

print("COMMIT TRACEABILITY: PASS")

expected SHA : 11064f15cb8d52e6d58d3612c6a535a8e4428adf
recorded SHA : {'11064f15cb8d52e6d58d3612c6a535a8e4428adf'}
COMMIT TRACEABILITY: PASS
